## [4.7. match 文](https://docs.python.org/ja/3.14/tutorial/controlflow.html#match-statements)

* switch文と表面的には似てて...
* どのcaseにもマッチしない場合はどの分岐も実行されません。


match 文は1つの式を指定し、その値と次に続く1つ以上のcaseブロックに指定されたパターンを比較します。この機能はCやJava、JavaScript(や他の多数の言語)のswitch文と表面的には似ていますが、RustやHaskellのパターンマッチングにより似ています。最初にマッチしたパターンのみが実行され、コンポーネント(シーケンスの要素やオブジェクトの属性)から値を取り出して変数に代入することもできます。どのcaseにもマッチしない場合はどの分岐も実行されません。

最も単純な形式は、対象の値に対して1つ以上のリテラルです:

In [2]:
import sys
print(sys.version)

3.14.4 (main, Apr  7 2026, 13:13:20) [Clang 21.0.0 (clang-2100.0.123.102)]


In [3]:
def http_error(status):
    match status:
        case 400:
            return "Bad request"
        case 404:
            return "Not found"
        case 418:
            return "I'm a teapot"
        case _:
            return "Something's wrong with the internet"

変数名 _ は ワイルドカード の働きをし、マッチに絶対失敗しません。

複数のリテラルを | ("or")を使用して組み合わせて1つのパターンにできます:

In [5]:
def http_error2(status):
    match status:
        case 401 | 403 | 404:
            return "Not allowed"
        case _:
            return "Unknown error"

パターンはアンパック代入ができ、変数に結びつけられます:



In [ ]:
# point(座標) は (x, y) のタプル
match point:
    case (0, 0):
        print("原点")
    case (0, y):
        print(f"y={y} 上の点")
    case (x, 0):
        print(f"x={x} 上の点")
    case (x, y):
        print(f"({x}, {y}) の点")
    case _:
        raise ValueError("座標ではない")

SyntaxError: invalid syntax (2998186398.py, line 2)

このコードは注意して見てください！ 最初のパターンには2つのリテラルがあり、上で示したリテラルパターンの拡張と考えることができます。 しかし次の2つのパターンはリテラルと変数の組み合わせのため、対象(point)から値を取り出して変数に 結びつけ ます。 4番目のパターンは2つの値を取り込みます。 これは、アンパック代入 (x, y) = point と概念的に似ています。

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

def where_is(point):
    match point:
        case Point(x=0, y=0):
            print("原点")
        case Point(x=0, y=y):
            print(f"Y={y} 上の点")
        case Point(x=x, y=0):
            print(f"X={x} 上の点")
        case Point():
            print("それ以外のどこか")
        case _:
            print("座標ではない")

SyntaxError: invalid syntax (2909124922.py, line 7)

いくつかの組み込みクラスでは位置引数が使用でき、属性の順番を提供します(例: データクラス)。クラスの __match_args__ 特殊属性によって、パターンの中で属性の明確な位置を定義することもできます。("x", "y")が設定された場合、以下のすべてのパターンは等価です(すべて属性 y が var 変数に結びつけられます):

```
Point(1, var)
Point(1, y=var)
Point(x=1, y=var)
Point(y=var, x=1)
```
おすすめのパターンの読み方は、パターンが、代入文の左辺に配置するものを拡張した形式であるとみなすことです。 これにより、どの変数になにが代入されるかが分かります。 単独の名前（上記の var など）だけがマッチ文で値が代入されます。ドット付きの名前（foo.bar など）、属性名（上記の x=、y= など ）、クラス名（名前の後ろの "(...)" によって判別される。上記の Point など）には値は代入されません。

パターンはいくらでも入れ子 (ネスト) にすることができます。例えば、 __match_args__ を追加した Point クラスのリストに対して次のようにマッチを行うことができます:


In [ ]:
class Point:
    __match_args__ = ('x', 'y')
    def __init__(self, x, y):
        self.x = x
        self.y = y
match points:
    case []:
        print("座標が存在しない")
    case [Point(0, 0)]:
        print("原点")
    case [Point(x, y)]:
        print(f"1つの座標 {x}, {y}")
    case [Point(0, y1), Point(0, y2)]:
        print(f"Y軸の {y1}, {y2} に2つの座標")
    case _:
        print("それ以外のどこか")

SyntaxError: invalid syntax (1078207986.py, line 6)

パターンに if 節を追加できます。これは "ガード" と呼ばれます。ガードがfalseの場合、match は次のcaseブロックの処理に移動します。ガードを評価する前に値が取り込まれることに注意してください:

In [ ]:
match point:
    case Point(x, y) if x == y:
        # 対角線上の点
        print(f"Y=X 上の点 at {x}")
    case Point(x, y):
        print(f"対角線上ではない")

SyntaxError: invalid syntax (3011991039.py, line 1)

(TODO: Skipping this part. Revisit to read this later)

この文のその他のいくつか重要な特徴:

アンパック代入のように、タプルとリストのパターンでは正確に同じ意味で、任意のシーケンスと一致します。重要な例外として、イテレーターや文字列ではマッチしません。

シーケンスパターンは拡張アンパックをサポート: [x, y, *rest] と (x, y, *rest) はアンパック代入として同じように動作します。* のあとの変数名は _ でもよく、そのため (x, y, *_) は最低でも2つのアイテムを持つシーケンスにマッチし、残りのアイテムは変数に結びつけられません。

マッピングパターン: {"bandwidth": b, "latency": l} は辞書から "bandwidth" と "latency" の値を取り込みます。シーケンスパターンとは異なり、それ以外のキーは無視されます。アンパッキングのような **rest もサポートされています（しかし、 **_ は冗長なため禁止されています）。

サブパターンでは as キーワードを使用して値を取り込みます:

（入力が2つのポイントのシーケンスである場合）

```python
case (Point(x1, y1), Point(x2, y2) as p2):
```

この例では入力から2番目の要素を p2 として取り込みます

ほとんどのリテラルは同一性を比較しますが、シングルトンの True、False、None では識別値を比較します。

Patterns may use named constants. These must be dotted names to prevent them from being interpreted as capture variables:

より詳細な説明と追加の例は PEP 636 にチュートリアル形式で記述してあります。

In [ ]:
from enum import Enum
class Color(Enum):
    RED = 'red'
    GREEN = 'green'
    BLUE = 'blue'

color = Color( input("色を入力してください (red, green, blue): ") )

match color:
    case Color.RED:
        print("赤")
    case Color.GREEN:
        print("緑")
    case Color.BLUE:
        print("青")

SyntaxError: invalid syntax (1985955621.py, line 9)